In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType,IntegerType,StructType,StructField,DateType ,DecimalType
from pyspark.sql.functions import col,desc,dense_rank,avg,when ,date_diff,current_date,to_date,sum,date_add

In [0]:
_schemaDefination = StructType(
                    [
                        StructField("first_name",StringType(),True),
                        StructField("last_name",StringType(),True),
                        StructField("job_title",StringType(),True),
                        StructField("dob",StringType(),True),
                        StructField("email",StringType(),True),
                        StructField("phone",StringType(),True),
                        StructField("salary",IntegerType(),True),
                        StructField("department_id",IntegerType(),True)
                    ]
)

In [0]:
# Redaing the data from the volumne 

employeesDf  =(
                        spark.read.
                        format("csv").
                        option("header",True).
                        option("inferSchema",True).
                        schema(_schemaDefination).
                        load("/Volumes/hands_on_catlog/default/raw_data/employee_records.csv")
        
)

In [0]:
# Departments schema  defination 
_deptSchema = StructType(

                [
                    StructField("department_id",IntegerType(),True),
                    StructField("department_name",StringType(),True),
                    StructField("description",StringType(),True),
                    StructField("city",StringType(),True),
                    StructField("state",StringType(),True),
                    StructField("country",StringType(),True),
                ]

)

In [0]:
# redaing the data from the  departments table 

departmentsDf = (
        spark.
        read.
        format("csv").
        option("header",True).
        schema(_deptSchema).
        load("/Volumes/hands_on_catlog/default/raw_data/department_data.csv")
)

In [0]:
departmentsDf.printSchema()

In [0]:
departmentsDf.limit(5).show()

In [0]:
# employyes whose salary is  more than 60000

employeesDf.filter("salary > 60000").show()

# joing the two different data frames 



In [0]:
# joining the two data frames 
(
    employeesDf.
    alias("t1").
    join(departmentsDf.alias("t2"),
         on=col("t1.department_id") == col("t2.department_id"),
         how = 'inner'
    )
).select (
      col("t1.first_name").alias("fname") , 
    col("t1.last_name").alias("lname"),
    col("t1.department_id").alias("departmentId"),
    col("t2.department_name").alias("deptName")
    ).show()

In [0]:
# finding the second highest salary in each dept 
from pyspark.sql.window import Window 

window_spec = Window.partitionBy("department_id").orderBy(col("salary").desc())

tempDf = employeesDf.withColumn("ranking",dense_rank().over(window_spec))
tempDf.filter("ranking==2").show()

In [0]:
# finding the department which is having the highest salary 

highestSalaryDept =(
                    employeesDf.
                    groupBy("department_id") .agg(
                    avg("salary").
                    alias("departmentsAvgSalary").cast(DecimalType(9,2))
                    )
                    
                    .orderBy(col("departmentsAvgSalary").desc()).limit(1)
)

In [0]:
# Top three earners per dept 
window_spec2 =Window.partitionBy(col("department_id")).orderBy(col("salary").desc())
topThreeDf = employeesDf.withColumn("deprtSalRanking",
                                    dense_rank().over(window_spec2)
                                    ).filter("deprtSalRanking <= 3").show()
    


In [0]:
# dedeuplication by email   keeping the  highest salaried one 

windowSpec3  =  Window.partitionBy(col("email")).orderBy(col("salary").desc())

emailDuplication = employeesDf.withColumn("ranking",dense_rank().over(windowSpec3)).filter("ranking  = 1")
emailDuplication.count()

In [0]:
# employees who are having high salary compared to there departments avg salary 
window_spec5 =Window.partitionBy(col("department_id"))

In [0]:
employeesDf.limit(10).show()

In [0]:
# caluclating the age of the employee 
employeesDf.withColumn("age",datediff(current_date(),to_date(col("dob"),"yyyy-mm-dd"))/365).show()

In [0]:
employeesDf.printSchema()

In [0]:
windowSpecCommulative = Window.partitionBy().orderBy(col("salary").desc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)
resultsDF = employeesDf.withColumn(
                        "cummlativeSalary",
                       sum(col("salary")).over(windowSpecCommulative)
                       ).limit(10).show()
                

In [0]:
resultsDF.limit(10).show()

In [0]:
employeesDf.createOrReplaceTempView("employeesDetails")

In [0]:
%sql
SELECT 
first_name , 
last_name ,
salary,
sum(salary) over( order by  first_name )
FROM employeesDetails

In [0]:
window_spec6 = Window.partitionBy("department_id").orderBy(col("first_name")).rowsBetween(Window.unboundedPreceding,Window.currentRow)

employeesDf.withColumn("runningSalaryDept",
                       sum(col("salary")).over(window_spec6)
                        ).select("first_name","last_name","runningSalaryDept").show()

In [0]:
%sql
select * from employeesDetails limit 10

In [0]:

from pyspark.sql.functions import sum, col 

from pyspark.sql.window import Window

window_spec = Window.partitionBy("department_id").orderBy(col("first_name").desc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)

employeesDf = employeesDf.withColumn("commulativeSum" , sum(col("salary")).over(window_spec))

employeesDf.select("commulativeSum").show()
                                                

In [0]:
employeesDf.withColumn("dob",date_add(col("dob"),)).show()

In [0]:
employeesDf.withColumn("dateAdd",date_diff(current_date(),col('dob'))).show("dateAdd")